# 16S CTF Analyses of the Longitudinal Acne Study

Date created: 10/18/2024

Notebook author: Britta De Pessemier 

Data analysis by: Tyler Myers, Britta De Pessemier, and Yang Chen

This notebook plots the following:
- Compositional Tensor Factorization for 16S data

In [8]:
# Import essential Python packages
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.gridspec import GridSpec
from matplotlib.patches import Ellipse
from matplotlib.colors import LinearSegmentedColormap

# Scientific and statistical packages
import scipy.stats as ss
from skbio import DistanceMatrix
from skbio.stats.ordination import OrdinationResults
from skbio.stats.distance import permanova

# BIOM and Qiime2 related packages
import biom
import qiime2 as q2
from biom import load_table
from qiime2 import Artifact
import h5py

import warnings
warnings.filterwarnings(
    "ignore",
    category=UserWarning,
    message="You passed a edgecolor/edgecolors .*"
)


In [9]:
# Function to draw an ellipse based on the covariance of the points
def draw_ellipse(mean, cov, ax, color, scale_factor=1.5):
    eigenvalues, eigenvectors = np.linalg.eigh(cov)
    angle = np.degrees(np.arctan2(*eigenvectors[:, 0][::-1]))
    width, height = 2 * np.sqrt(eigenvalues) * scale_factor

    # Add ellipse to the plot
    ell = Ellipse(
        xy=mean,
        width=width,
        height=height,
        angle=angle,
        edgecolor=color,
        facecolor=color,
        alpha=0.2,
        linewidth=0.5
    )
    ax.add_patch(ell)

## Compositional Tensor Factorization (CTF) analysis
Analysis per participant skin site location (C1/C2 for healthy and C1/C2/C3 for acne) 

In [10]:
# Define the color palette for the groups
group_colors = {
    "Healthy": "#000096",  # Color for Healthy
    "Acne_L": "#ff676c",   # Color for Acne Lesional
    "Acne_NL": "#17c6d5"   # Color for Acne Non-lesional
}

# Map the old group names to new ones for the legend
group_name_mapping = {
    "Healthy": "Healthy",
    "Acne_L": "Acne Lesional",
    "Acne_NL": "Acne Non-lesional"
}

# List of folders to iterate over and corresponding titles and filenames
folders = [
    ('174950', 'V1-V3_CTF', 'CTF - 16S rRNA (V1-V3)'),
    ('174951', 'V4_CTF', 'CTF - 16S rRNA (V4)')
]

# Loop over the folders
for folder, file_suffix, plot_title in folders:
    # print(f"Processing folder: {folder}")

    # Load the distance matrix artifact
    table = Artifact.load(f'../Data/16S/CTF/ctf-results_{folder}/distance_matrix.qza')

    # View as a skbio DistanceMatrix object
    distance_matrix = table.view(DistanceMatrix)

    # Convert the skbio DistanceMatrix to a Pandas DataFrame
    data = pd.DataFrame(distance_matrix.data, index=distance_matrix.ids, columns=distance_matrix.ids)
    # print(data)

    # Load the subject_biplot artifact
    biplot = Artifact.load(f'../Data/16S/CTF/ctf-results_{folder}/subject_biplot.qza')

    # View as an OrdinationResults object
    ordination = biplot.view(OrdinationResults)

    # Extract the sample coordinates as a DataFrame
    sample_coordinates = pd.DataFrame(ordination.samples, index=ordination.samples.index)

    # Extract the feature (or variable) coordinates as a DataFrame
    feature_coordinates = pd.DataFrame(ordination.features, index=ordination.features.index)

    # print("Sample Coordinates:")
    # print(sample_coordinates)
    # print("\nFeature Coordinates:")
    # print(feature_coordinates)

    # Extract the proportion explained
    proportion_explained = ordination.proportion_explained

    # Convert to percentages for PC1 and PC2
    pc1_explained_variance = proportion_explained[0] * 100
    pc2_explained_variance = proportion_explained[1] * 100

    # print(f"PC1 explains {pc1_explained_variance:.2f}% of the variance")
    # print(f"PC2 explains {pc2_explained_variance:.2f}% of the variance")

    # Rename the sample coordinates DataFrame to spca_df
    spca_df = sample_coordinates

    # Load the metadata from the subject-metadata.tsv file
    metadata_df = pd.read_csv(f'../Data/16S/CTF/ctf-results_{folder}/subject-metadata.tsv', sep='\t')

    # Map the index of spca_df (sample names) to the SampleID in metadata_df to get the 'group'
    spca_df["Group"] = spca_df.index.map(metadata_df.set_index("#SampleID")["group"])

    # Rename the columns of spca_df for clarity (PC1, PC2, PC3)
    spca_df.columns = ['PC1', 'PC2', 'PC3', 'Group']
    # print(spca_df)

    # Plotting
    fig, ax = plt.subplots(1, 1, figsize=(7, 10))

    # Create scatter plot with different colors based on the Group column
    for group_name, group in spca_df.groupby('Group'):
        sns.scatterplot(
            data=group,
            x="PC1",
            y="PC2",
            color=group_colors[group_name],
            edgecolor=group_colors[group_name],
            alpha=1,
            linewidth=0.5,
            ax=ax,
            label=group_name_mapping[group_name],
            s=100
        )

        # Calculate mean and covariance matrix for the group
        points = group[["PC1", "PC2"]].values
        mean = np.mean(points, axis=0)
        cov = np.cov(points, rowvar=False)

        # Draw the ellipse for the group (with scale factor applied)
        draw_ellipse(mean, cov, ax, group_colors[group_name], scale_factor=2)

        # Add a subtle star (8-pointed) at the mean location for each group
        ax.scatter(mean[0], mean[1], color=group_colors[group_name], marker=(8, 2, 0), s=300, edgecolor='black', zorder=5, linewidth=0.5)

    # Add axis labels and include the explained variance percentages
    ax.set_xlabel(f'PC1 ({pc1_explained_variance:.2f}%)', fontsize=18)
    ax.set_ylabel(f'PC2 ({pc2_explained_variance:.2f}%)', fontsize=18)

    # Simplify the ticks to avoid overcrowding
    yticklabels = [-0.4, -0.2, 0.0, 0.2, 0.4, 0.6]
    ax.set_yticks(yticklabels)
    ax.set_yticklabels(yticklabels, fontsize=18)

    xticklabels = [-0.4, -0.2, 0.0, 0.2, 0.4, 0.6]
    ax.set_xticks(xticklabels)
    ax.set_xticklabels(xticklabels, fontsize=18)

    # Set plot title dynamically
    if plot_title == 'CTF - 16S rRNA (V1-V3)':
        new_plot_title = '16S rRNA (V1-V3) CTF'
    elif plot_title == 'CTF - 16S rRNA (V4)':
        new_plot_title = '16S rRNA (V4) CTF'

    plt.title(new_plot_title, fontsize=20)
    
    # Set the thickness of the frame (axes spines)
    for spine in ax.spines.values():
        spine.set_linewidth(1.2)
        
    # Customize legend
    ax.legend(
        frameon=False,
        fontsize=18,
        # title="Group",
        title_fontsize=18
    )

    # Adjust plot style and save
    ax.spines["right"].set_visible(True)
    ax.spines["top"].set_visible(True)

    plt.tight_layout()
    plt.savefig(f"../Figures/16S_Figures/CTF/{file_suffix}.png", dpi=600)

## CTF colored by severity group

In [11]:
# Create scatter plot with different colors based on the Group column
for group_name, group in spca_df.groupby('Group'):
    # Calculate mean and covariance matrix for the group
    points = group[["PC1", "PC2"]].values

    # Debugging: Check the shape of points
    # print(f"Points shape for group '{group_name}': {points.shape}")

    if points.shape[0] < 2:  # Ensure at least 2 points for covariance
        print(f"Not enough points to compute covariance for group '{group_name}'")
        continue  # Correctly skip this group if not enough points

    # Calculate mean and covariance matrix for the group
    mean = np.mean(points, axis=0)
    cov = np.cov(points, rowvar=False)

    # Draw the ellipse for the group (with scale factor applied)
    draw_ellipse(mean, cov, ax, group_colors[group_name], scale_factor=2)

    # Add a subtle star (8-pointed) at the mean location for each group
    ax.scatter(mean[0], mean[1], color=group_colors[group_name], marker=(8, 2, 0), s=100, edgecolor='black', zorder=5, linewidth=0.5)


In [12]:
# Define the color palette for the groups
group_colors = {
    "absent Healthy": "#4d4db3",    # Lighter blue for absent Healthy
    "absent Acne_NL": "#a5e5f0",    # Light cyan for absent absent Acne Non-lesional
    "low Acne_NL": "#17c6d5",       # Cyan blue for low Acne Non-lesional
    "Acne_L": "#ff676c",            # Coral pink for Acne Lesional
    "low Acne_L": "#ffb3b8",        # Soft pink for low Acne Lesional
    "moderate Acne_L": "#ff3b4d",   # Bright red-pink for moderate Acne Lesional
    "high Acne_L": "#960000"        # Deep maroon for high Acne Lesional
}

# Map the old group names to new ones for the legend
group_name_mapping = {
    "absent Healthy": "absent Healthy",
    "absent Acne_NL": "absent Acne non-lesional",
    "low Acne_NL": "low Acne non-lesional", 
    "low Acne_L": "low Acne lesional",
    "moderate Acne_L": "moderate Acne lesional", 
    "high Acne_L": "high Acne lesional"
}

# List of folders to iterate over and corresponding titles and filenames
folders = [
    ('174950', 'V1-V3_CTF', 'CTF - 16S rRNA (V1-V3)'),
    ('174951', 'V4_CTF', 'CTF - 16S rRNA (V4)')
]

# Loop over the folders
for folder, file_suffix, plot_title in folders:
    # print(f"Processing folder: {folder}")
    
    # Load the distance matrix artifact
    table = Artifact.load(f'../Data/16S/CTF/ctf-results_{folder}/distance_matrix.qza')
    
    # View as a skbio DistanceMatrix object
    distance_matrix = table.view(DistanceMatrix)
    
    # Convert the skbio DistanceMatrix to a Pandas DataFrame
    data = pd.DataFrame(distance_matrix.data, index=distance_matrix.ids, columns=distance_matrix.ids)
    
    # Display the DataFrame
    # print(data)
    
    # Load the subject_biplot artifact
    biplot = Artifact.load(f'../Data/16S/CTF/ctf-results_{folder}/subject_biplot.qza')
    
    # View as an OrdinationResults object
    ordination = biplot.view(OrdinationResults)
    
    # Extract the sample coordinates as a DataFrame
    sample_coordinates = pd.DataFrame(ordination.samples, index=ordination.samples.index)
    
    # Extract the feature (or variable) coordinates as a DataFrame
    feature_coordinates = pd.DataFrame(ordination.features, index=ordination.features.index)
    
    # Display the DataFrames
    # print("Sample Coordinates:")
    # print(sample_coordinates)
    # print("\nFeature Coordinates:")
    # print(feature_coordinates)
    
    # Extract the proportion explained
    proportion_explained = ordination.proportion_explained
    
    # Convert to percentages for PC1 and PC2
    pc1_explained_variance = proportion_explained[0] * 100
    pc2_explained_variance = proportion_explained[1] * 100
    
    # print(f"PC1 explains {pc1_explained_variance:.2f}% of the variance")
    # print(f"PC2 explains {pc2_explained_variance:.2f}% of the variance")
    
    # Rename the sample coordinates DataFrame to spca_df
    spca_df = sample_coordinates
    
    # Load the metadata from the subject-metadata.tsv file
    metadata_df = pd.read_csv(f'../Data/16S/CTF/ctf-results_{folder}/subject-metadata_severity_group.tsv', sep='\t')
    
    # Map the index of spca_df (sample names) to the SampleID in metadata_df to get the 'group'
    spca_df["Group"] = spca_df.index.map(metadata_df.set_index("#SampleID")["severity_group"])
    
    # Rename the columns of spca_df for clarity (PC1, PC2, PC3)
    spca_df.columns = ['PC1', 'PC2', 'PC3', 'Group']
    
    # Display the updated spca_df
    # print(spca_df)
    
    # Plotting
    mm = 1/25.4
    fig, ax = plt.subplots(1, 1, figsize=(90*mm, 110*mm))
    
    # Create scatter plot with different colors based on the Group column
    for group_name, group in spca_df.groupby('Group'):
        # Calculate points for the group
        points = group[["PC1", "PC2"]].values
        
        # Debugging: Check the shape of points
        # print(f"Points shape for group '{group_name}': {points.shape}")

        # Check if the group has enough points for covariance calculation
        if points.shape[0] < 2:
            print(f"Not enough points to compute covariance for group '{group_name}'. Skipping this group.")
            continue  # Skip this group if not enough points

        # Create scatter plot for the group
        sns.scatterplot(
            data=group,
            x="PC1",
            y="PC2",
            facecolor=group_colors[group_name],  # Semi-transparent fill color
            edgecolor=group_colors[group_name],  # Colored border
            alpha=0.8,  # Transparency for the fill
            linewidth=0.5,  # Thickness of the borders
            ax=ax,
            label=group_name_mapping[group_name]
        )

        # Calculate mean and covariance matrix for the group
        mean = np.mean(points, axis=0)
        cov = np.cov(points, rowvar=False)

        # Draw the ellipse for the group (with scale factor applied)
        draw_ellipse(mean, cov, ax, group_colors[group_name], scale_factor=2)

        # Add a subtle star (8-pointed) at the mean location for each group
        ax.scatter(mean[0], mean[1], color=group_colors[group_name], marker=(8, 2, 0), s=100, edgecolor='black', zorder=5, linewidth=0.5)

    # Add axis labels and include the explained variance percentages
    ax.set_xlabel(f'PC1 ({pc1_explained_variance:.2f}%)', fontsize=7)
    ax.set_ylabel(f'PC2 ({pc2_explained_variance:.2f}%)', fontsize=7)
    
    # Simplify the ticks to avoid overcrowding
    yticklabels = [-0.4, -0.2, 0.0, 0.2, 0.4, 0.6]
    ax.set_yticks(yticklabels)
    ax.set_yticklabels(yticklabels, fontsize=7)
    
    xticklabels = [-0.4, -0.2, 0.0, 0.2, 0.4, 0.6]
    ax.set_xticks(xticklabels)
    ax.set_xticklabels(xticklabels, fontsize=7)
    
    # Add the title
    ax.text(0.5, 1.025, plot_title, fontsize=8, va='center', ha='center', transform=ax.transAxes)

    # Customize legend
    plt.legend(
        frameon=False,
        fontsize=7,
        title_fontsize=7
    )
    
    # Adjust plot style and save
    ax.spines["right"].set_visible(True)
    ax.spines["top"].set_visible(True)
    
    plt.tight_layout()
    plt.savefig(f"../Figures/16S_Figures/CTF/{file_suffix}_severity_group.png", dpi=600)


Not enough points to compute covariance for group 'high Acne_L'. Skipping this group.
Not enough points to compute covariance for group 'high Acne_L'. Skipping this group.


## CTF plots with continuous severity scores

In [13]:
# Severity group 
metadata_df = pd.read_csv(f'../Metadata/metadata_final_22102024.tsv', sep='\t')

# Group by 'subject_ID_CC' and calculate the mean of 'local_lesion_severity'
mean_severity = metadata_df.groupby('subject_ID_CC')['local_lesion_severity'].transform('mean')

# Add this mean value as a new column in metadata_df
metadata_df['local_lesion_severity_mean_ID_CC'] = mean_severity

mf = metadata_df.groupby('subject_ID_CC').agg({'group':'first','local_lesion_severity_mean_ID_CC':'first'})

mf.index.name = '#SampleID'

mf.to_csv('../Data/16S/CTF/ctf-results_174951/subject-metadata_subject_ID_mean_severity.tsv', sep='\t')

In [14]:
# Define the color gradient for 'local_lesion_severity_mean_ID_CC' as a colormap
custom_colors = ["#423fa6", "#67b2be", "#ededed", "#df7963", "#ca0000", "#960000"]
severity_cmap = LinearSegmentedColormap.from_list("severity_gradient", custom_colors)

# List of folders to iterate over and corresponding titles and filenames
folders = [
    ('174950', 'V1-V3_CTF', 'CTF - 16S rRNA (V1-V3)'),
    ('174951', 'V4_CTF', 'CTF - 16S rRNA (V4)')
]

# Loop over the folders
for folder, file_suffix, plot_title in folders:
    # print(f"Processing folder: {folder}")
    
    # Load the distance matrix artifact
    table = Artifact.load(f'../Data/16S/CTF/ctf-results_{folder}/distance_matrix.qza')
    distance_matrix = table.view(DistanceMatrix)
    data = pd.DataFrame(distance_matrix.data, index=distance_matrix.ids, columns=distance_matrix.ids)
    # print(data)
    
    # Load the subject biplot artifact
    biplot = Artifact.load(f'../Data/16S/CTF/ctf-results_{folder}/subject_biplot.qza')
    ordination = biplot.view(OrdinationResults)
    
    # Extract sample coordinates and proportion explained
    spca_df = pd.DataFrame(ordination.samples, index=ordination.samples.index)
    proportion_explained = ordination.proportion_explained
    pc1_explained_variance = proportion_explained[0] * 100
    pc2_explained_variance = proportion_explained[1] * 100
    
    # Load the metadata
    metadata_df = pd.read_csv(f'../Data/16S/CTF/ctf-results_{folder}/subject-metadata_mean_severity.tsv', sep='\t')
    
    # Map the local lesion severity mean score for each subject
    spca_df["local_lesion_severity_mean_ID_CC"] = spca_df.index.map(metadata_df.set_index("#SampleID")["local_lesion_severity_mean_ID_CC"])
    
    # Rename columns for clarity (PC1, PC2, PC3)
    spca_df.columns = ['PC1', 'PC2', 'PC3', 'local_lesion_severity_mean_ID_CC']
    
    # Plotting
    mm = 1 / 25.4
    fig, ax = plt.subplots(1, 1, figsize=(90 * mm, 110 * mm))
    
    # Create a scatter plot with a gradient color based on 'local_lesion_severity_mean_ID_CC'
    sc = ax.scatter(
        spca_df['PC1'],
        spca_df['PC2'],
        c=spca_df['local_lesion_severity_mean_ID_CC'],
        cmap=severity_cmap,
        edgecolor='k',
        linewidths= 0.3,
        alpha=0.8,
        s=30  # Adjust size as needed
    )
    
    # Add color bar
    cbar = plt.colorbar(sc, ax=ax)
    cbar.set_label('Mean Local Lesion Severity per subjectID\'s C-zone ', fontsize=7)
    cbar.ax.tick_params(labelsize=6)
    
    # Add axis labels and include the explained variance percentages
    ax.set_xlabel(f'PC1 ({pc1_explained_variance:.2f}%)', fontsize=7)
    ax.set_ylabel(f'PC2 ({pc2_explained_variance:.2f}%)', fontsize=7)
    
    # Set simplified ticks
    yticklabels = [-0.4, -0.2, 0.0, 0.2, 0.4, 0.6]
    ax.set_yticks(yticklabels)
    ax.set_yticklabels(yticklabels, fontsize=7)
    
    xticklabels = [-0.4, -0.2, 0.0, 0.2, 0.4, 0.6]
    ax.set_xticks(xticklabels)
    ax.set_xticklabels(xticklabels, fontsize=7)
    
    # Add the title
    ax.text(0.5, 1.025, plot_title, fontsize=8, va='center', ha='center', transform=ax.transAxes)

    # Style the plot
    ax.spines["right"].set_visible(False)
    ax.spines["top"].set_visible(False)
    
    plt.tight_layout()
    plt.savefig(f"../Figures/16S_Figures/CTF/{file_suffix}_local_lesion_severity_mean_ID_CC_continuous.png", dpi=600)